# 00 - Uniformation of the dataset
This notebook standardizes schemas and identifiers across GSE69914, GSE287331, and GSE225845, ensuring consistent columns (`id_tissue`, `label`) and lightweight integrity checks.
It converts GSE225845 TXT exports (normal/adjacent/tumor) to LZ4-compressed, float32 Parquet via streaming writes, then vertically merges them and assigns labels (0/1/2).
It aligns beta matrices with phenotype tables by joining `basenames` ↔ `idat_basename`, filters pheno to samples present in β, and saves the cleaned outputs.
Finally, it performs quick shape/header inspections and detects one-to-many basename mismatches, emitting a CSV report when ambiguities exist.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   POLITECNICO DI TORINO — MSc Mathematical Engineering           ║
# ║   THESIS: Advanced Study of Epigenetic Mechanisms in Neoplasms   ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Author:       Elisabetta Roviera (s328422)                       ║
# ║ Supervisors:  Dr. Sandro Gambino, Prof. Alfredo Benso            ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Notebook:     00-uniformation-of-the-gse                         ║
# ║ Description: : Standardize schemas id_tissue, label for beta and ║
# ║                pheno dataset                                     ║
# ║ Dataset(s):   GSE69914, GSE287331, GSE225845                     ║
# ║ Repository:   github.com/elisabettaroviera/THESIS                ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Date: 12-Nov-2025 | Python 3.11.13                               ║
# ╚══════════════════════════════════════════════════════════════════╝


### Libraries

In [8]:
import polars as pl
from pathlib import Path
import os, gc, csv
from typing import List, Dict, Tuple
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


In [1]:
# COLS NAME
DATA_PATHS = {
    "GSE69914_beta":  "/kaggle/input/gse69914-parquet/GSE69914.parquet",
    "GSE287331_beta": "/kaggle/input/gse287331-lz4-parquet/GSE287331_lz4.parquet",
    "GSE225845_beta": "/kaggle/input/gse225845-parquet/GSE225845.parquet",
    "GSE69914_pheno": "/kaggle/input/pheno-gse69914/pheno_GSE69914_lz4.parquet",
    "GSE287331_pheno":"/kaggle/input/pheno-gse287331/pheno_GSE287331.parquet",
    "GSE225845_pheno":"/kaggle/input/pheno-gse225845/pheno_GSE225845.parquet",
}

# INSPECTION UTILITIES
def show_schema_info(dataset_name: str, path: str):
    """Print dataset schema summary depending on type (beta vs pheno)."""
    is_pheno = "pheno" in dataset_name.lower()
    df = pl.read_parquet(path, n_rows=6 if not is_pheno else None)

    print(f"\n{'='*70}")
    print(f"[INFO] {dataset_name}")
    print(f"Path: {path}")
    print(f"Shape: {df.shape}")

    if is_pheno:
        print("\nColumns:")
        for col in df.columns:
            print(f"  - {col}")
    else:
        print("\nFirst 6 columns:")
        print("  " + ", ".join(df.columns[:6]))

for name, path in DATA_PATHS.items():
    show_schema_info(name, path)



[INFO] GSE69914_beta
Path: /kaggle/input/gse69914-parquet/GSE69914.parquet
Shape: (6, 485514)

First 6 columns:
  id_tissue, label, cg00000029, cg00000108, cg00000109, cg00000165

[INFO] GSE287331_beta
Path: /kaggle/input/gse287331-lz4-parquet/GSE287331_lz4.parquet
Shape: (6, 866554)

First 6 columns:
  id_sample, cg00000029, cg00000103, cg00000109, cg00000155, cg00000158

[INFO] GSE225845_beta
Path: /kaggle/input/gse225845-parquet/GSE225845.parquet
Shape: (6, 750429)

First 6 columns:
  accession_num, basenames, label, cg18478105, cg09835024, cg14361672

[INFO] GSE69914_pheno
Path: /kaggle/input/pheno-gse69914/pheno_GSE69914_lz4.parquet
Shape: (407, 10)

Columns:
  - sample_id
  - label
  - sentrix_id
  - slide_id
  - er
  - pr
  - her2
  - ki67
  - group
  - batch

[INFO] GSE287331_pheno
Path: /kaggle/input/pheno-gse287331/pheno_GSE287331.parquet
Shape: (446, 5)

Columns:
  - geo_accession
  - sample_name
  - tissue_type_raw
  - label_3class
  - idat_basename

[INFO] GSE225845_pheno

In [7]:
# CHECK NO LABEL IN 287331
BETA_PATH = "/kaggle/input/gse287331-lz4-parquet/GSE287331_lz4.parquet"

# Load only header (no need to read all CpGs)
df_head = pl.read_parquet(BETA_PATH, n_rows=5)

print(f"[INFO] GSE287331_beta — shape: {df_head.shape}")
print(f"[INFO] Columns detected (first 10): {df_head.columns[:10]}")
print(f"[INFO] Columns detected (last 10): {df_head.columns[-10:]}")


if "label" in df_head.columns:
    print("\n✅ 'label' column found.")
else:
    print("\n⚠️  'label' column NOT found in GSE287331_beta.")

[INFO] GSE287331_beta — shape: (5, 866554)
[INFO] Columns detected (first 10): ['id_sample', 'cg00000029', 'cg00000103', 'cg00000109', 'cg00000155', 'cg00000158', 'cg00000165', 'cg00000221', 'cg00000236', 'cg00000289']
[INFO] Columns detected (last 10): ['rs7746156', 'rs798149', 'rs845016', 'rs877309', 'rs9292570', 'rs9363764', 'rs939290', 'rs951295', 'rs966367', 'rs9839873']

⚠️  'label' column NOT found in GSE287331_beta.


In [9]:
# SAMPLE_ID -> ID_TISSUE IN 69914
PHENO_IN  = "/kaggle/input/pheno-gse69914/pheno_GSE69914_lz4.parquet"
PHENO_OUT = "/kaggle/working/pheno_GSE69914_idtissue_lz4.parquet"

# --- Load pheno ---
ph = pl.read_parquet(PHENO_IN)
print(f"[INFO] Loaded pheno: shape={ph.shape}")

# --- Rename column if needed ---
if "sample_id" in ph.columns:
    ph = ph.rename({"sample_id": "id_tissue"})
    print("✅ Renamed 'sample_id' → 'id_tissue'.")
else:
    print("ℹ️  Column 'sample_id' not found — no rename applied.")

# --- Verify ---
print(f"[CHECK] Columns: {', '.join(ph.columns)}")

# --- Write new file ---
Path(PHENO_OUT).parent.mkdir(parents=True, exist_ok=True)
ph.write_parquet(PHENO_OUT, compression="lz4", statistics=True)
print(f"✅ Saved updated pheno → {PHENO_OUT}")


[INFO] Loaded pheno: shape=(407, 10)
✅ Renamed 'sample_id' → 'id_tissue'.
[CHECK] Columns: id_tissue, label, sentrix_id, slide_id, er, pr, her2, ki67, group, batch
✅ Saved updated pheno → /kaggle/working/pheno_GSE69914_idtissue_lz4.parquet


In [10]:
# CHECK ALL WELL DONE
PHENO_PATH = "/kaggle/working/pheno_GSE69914_idtissue_lz4.parquet"

# --- Load pheno ---
ph = pl.read_parquet(PHENO_PATH)

# --- Summary ---
print(f"[INFO] Loaded pheno from: {PHENO_PATH}")
print(f"[INFO] Shape: {ph.shape}")

print("\n[INFO] Columns:")
for c in ph.columns:
    print(f"  - {c}")
    

[INFO] Loaded pheno from: /kaggle/working/pheno_GSE69914_idtissue_lz4.parquet
[INFO] Shape: (407, 10)

[INFO] Columns:
  - id_tissue
  - label
  - sentrix_id
  - slide_id
  - er
  - pr
  - her2
  - ki67
  - group
  - batch


In [ ]:
# GSE287331
# Paths 
BETA_IN  = "/kaggle/input/1-gse287331-parquet/GSE287331.parquet"
PHENO_IN = "/kaggle/input/1-pheno-gse287331-parquet/pheno_GSE287331.parquet"
BETA_OUT = "/kaggle/working/GSE287331_beta_idtissue_only.parquet"

# 1) Load phenotype data
ph = pl.read_parquet(PHENO_IN)
print(f"[INFO] Loaded pheno: shape={ph.shape}")
print("[CHECK] Columns in pheno:", ph.columns)

# Must contain at least these columns
for col in ["id_tissue", "idat_basename", "label"]:
    if col not in ph.columns:
        raise ValueError(f"'{col}' not found in pheno.")

# Minimal pheno for join: use idat_basename as the key
ph_min = ph.select(["idat_basename", "id_tissue", "label"])

# 2) Lazy-load β (Beta values)
beta_lf = pl.scan_parquet(BETA_IN)
beta_cols = beta_lf.columns
print("[CHECK] Columns in β (first 10):", beta_cols[:10])

if "id_sample" not in beta_cols:
    raise ValueError("'id_sample' not found in β (expected to match idat_basename).")

# 3) Join id_sample (β) <-> idat_basename (pheno)
joined_lf = beta_lf.join(
    ph_min.lazy().rename({"idat_basename": "id_sample"}),
    on="id_sample",
    how="left",
)

# 4) Build the final β matrix: id_tissue | label | CpG
joined = joined_lf.collect(streaming=True)
print(f"[INFO] Joined β shape: {joined.shape}")
print("[CHECK] Columns after join (first 10):", joined.columns[:10])

# Check that id_tissue and label are present
for col in ["id_tissue", "label"]:
    if col not in joined.columns:
        raise ValueError(f"'{col}' not found after join.")

# Identify CpGs = all columns that are NOT metadata
meta_cols = {"id_sample", "id_tissue", "label"}
cpg_cols = [c for c in joined.columns if c not in meta_cols]

print(f"[INFO] Number of CpG columns: {len(cpg_cols)}")

# Select the final column order
beta_final = joined.select(["id_tissue", "label", *cpg_cols])

print(f"[INFO] Final β shape: {beta_final.shape}")
print("[CHECK] First 5 columns:", beta_final.columns[:5])
print("[CHECK] Last 5 columns:", beta_final.columns[-5:])

# 5) Save
beta_final.write_parquet(
    BETA_OUT,
    compression="lz4",
    statistics=True,
)
print(f"✅ Saved final β → {BETA_OUT}")


[INFO] Loaded pheno: shape=(446, 5)
[CHECK] First 5 values in 'geo_accession': ['GSM8744013', 'GSM8744014', 'GSM8744015', 'GSM8744016', 'GSM8744017']
✅ Renamed in pheno: {'geo_accession': 'id_tissue', 'label_3class': 'label'}
✅ Saved updated pheno → /kaggle/working/pheno_GSE287331_idtissue_lz4.parquet


/tmp/ipykernel_48/3423832348.py:48: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  beta_cols = beta_lf.columns


✅ Renamed in β: 'id_sample' → 'id_tissue'


/tmp/ipykernel_48/3423832348.py:62: DeprecationWarning: The argument `streaming=True` is deprecated and is being replaced by the `engine` argument.
  joined = joined_lf.collect(streaming=True)
/tmp/ipykernel_48/3423832348.py:62: DeprecationWarning: The old streaming engine is being deprecated and will soon be replaced by the new streaming engine. Starting Polars version 1.23.0 and until the new streaming engine is released, the old streaming engine may become less usable. For people who rely on the old streaming engine, it is suggested to pin your version to before 1.23.0.

More information on the new streaming engine: https://github.com/pola-rs/polars/issues/20947
  joined = joined_lf.collect(streaming=True)


[INFO] Labeled β shape: (446, 866555)
[CHECK] Last 5 columns after join (should include 'label'): ['rs939290', 'rs951295', 'rs966367', 'rs9839873', 'label']
✅ Wrote labeled β → /kaggle/working/GSE287331_beta_labeled_lz4.parquet


In [12]:
# CHECK ALL WELL DONE -pheno
PHENO_PATH = "/kaggle/working/pheno_GSE287331_idtissue_lz4.parquet"

# --- Load pheno ---
ph = pl.read_parquet(PHENO_PATH)

# --- Summary ---
print(f"[INFO] Loaded pheno from: {PHENO_PATH}")
print(f"[INFO] Shape: {ph.shape}")

print("\n[INFO] Columns:")
for c in ph.columns:
    print(f"  - {c}")

[INFO] Loaded pheno from: /kaggle/working/pheno_GSE287331_idtissue_lz4.parquet
[INFO] Shape: (446, 5)

[INFO] Columns:
  - id_tissue
  - sample_name
  - tissue_type_raw
  - label
  - idat_basename


In [15]:
# CHECK ALL WELL DONE - dataset
DATA_PATH = "/kaggle/working/GSE287331_beta_labeled_lz4.parquet"

# --- Load pheno ---
dataset = pl.read_parquet(DATA_PATH)

# --- Summary ---
print(f"[INFO] Loaded pheno from: {DATA_PATH}")
print(f"[INFO] Shape: {dataset.shape}")

print(f"[INFO] Columns detected (first 10): {dataset.columns[:10]}")
print(f"[INFO] Columns detected (last 10): {dataset.columns[-10:]}")

[INFO] Loaded pheno from: /kaggle/working/GSE287331_beta_labeled_lz4.parquet
[INFO] Shape: (446, 866555)
[INFO] Columns detected (first 10): ['id_tissue', 'cg00000029', 'cg00000103', 'cg00000109', 'cg00000155', 'cg00000158', 'cg00000165', 'cg00000221', 'cg00000236', 'cg00000289']
[INFO] Columns detected (last 10): ['rs798149', 'rs845016', 'rs877309', 'rs9292570', 'rs9363764', 'rs939290', 'rs951295', 'rs966367', 'rs9839873', 'label']


In [17]:
# CHECK 225845
# --- Paths (from your DATA_PATHS memory) ---
BETA_PATH  = "/kaggle/input/gse225845-parquet/GSE225845.parquet"
PHENO_PATH = "/kaggle/input/pheno-gse225845/pheno_GSE225845.parquet"

# --- Helper: resolve requested column names case-insensitively ---
def resolve_columns(parquet_path: str, requested: list[str]) -> list[str]:
    cols = pl.read_parquet(parquet_path, n_rows=0).columns  # header only
    lower_map = {c.lower(): c for c in cols}
    resolved = []
    for r in requested:
        hit = lower_map.get(r.lower())
        if not hit:
            print(f"⚠️  Column '{r}' not found in {parquet_path}. Available (sample): {cols[:8]} ...")
        else:
            resolved.append(hit)
    return resolved

# BETA: accession_num, basenames, label
beta_req = ["accession_num", "basenames", "label"]
beta_cols = resolve_columns(BETA_PATH, beta_req)

if beta_cols:
    beta_first5 = (
        pl.scan_parquet(BETA_PATH)
          .select(beta_cols)     # no reordering beyond what's requested
          .head(5)
          .collect()
    )
    print(f"[INFO] BETA first 5 rows for {beta_cols}:")
    for c in beta_first5.columns:
        print(f"  - {c}: {beta_first5[c].to_list()}")
else:
    print("[INFO] No beta columns resolved; nothing to print.")

# PHENO: geo_accession, sample_name, source_name, label_3class, sample_type_raw, tissue_type_raw, idat_basename
pheno_req = ["geo_accession", "sample_name", "source_name", "label_3class", "sample_type_raw", "tissue_type_raw", 
            "idat_basename", ]
pheno_cols = resolve_columns(PHENO_PATH, pheno_req)

if pheno_cols:
    pheno_first5 = (
        pl.scan_parquet(PHENO_PATH)
          .select(pheno_cols)    # no reordering beyond what's requested
          .head(5)
          .collect()
    )
    print(f"[INFO] PHENO first 5 rows for {pheno_cols}:")
    for c in pheno_first5.columns:
        print(f"  - {c}: {pheno_first5[c].to_list()}")
else:
    print("[INFO] No pheno columns resolved; nothing to print.")


[INFO] BETA first 5 rows for ['accession_num', 'basenames', 'label']:
  - accession_num: ['12598', '12919', '12489', '12367', '12616']
  - basenames: ['203723060067_R04C01', '203723060067_R08C01', '204425700003_R01C01', '204425700003_R02C01', '204425700003_R03C01']
  - label: [0, 0, 0, 0, 0]
[INFO] PHENO first 5 rows for ['geo_accession', 'sample_name', 'source_name', 'label_3class', 'sample_type_raw', 'tissue_type_raw', 'idat_basename']:
  - geo_accession: ['GSM7057513', 'GSM7057514', 'GSM7057515', 'GSM7057516', 'GSM7057517']
  - sample_name: ['Genomic DNA from breast tissue 92-05 normal', 'Genomic DNA from breast tissue 92-6 normal', 'Genomic DNA from breast tissue 92-7 normal', 'Genomic DNA from breast tissue 92-8 normal', 'Genomic DNA from breast tissue 92-9 normal']
  - source_name: ['frozen normal breast tissue', 'frozen normal breast tissue', 'frozen normal breast tissue', 'frozen normal breast tissue', 'frozen normal breast tissue']
  - label_3class: [0, 0, 0, 0, 0]
  - sample_

In [18]:
# CORRECTION ON PHENO 225845
PHENO_IN  = "/kaggle/input/pheno-gse225845/pheno_GSE225845.parquet"
PHENO_OUT = "/kaggle/working/pheno_GSE225845_idtissue_lz4.parquet"

# --- Load ---
ph = pl.read_parquet(PHENO_IN)
print(f"[INFO] Loaded pheno: shape={ph.shape}")

# --- Rename (no reordering) ---
rename_map = {}
if "geo_accession" in ph.columns:
    rename_map["geo_accession"] = "id_tissue"
if "label_3class" in ph.columns:
    rename_map["label_3class"] = "label"

if rename_map:
    ph = ph.rename(rename_map)
    print(f"✅ Renamed: {rename_map}")
else:
    print("ℹ️  Nothing to rename: expected columns not found.")

# --- Quick verification ---
print("[CHECK] Columns now:")
print(", ".join(ph.columns))

# --- Save to /kaggle/working ---
Path(PHENO_OUT).parent.mkdir(parents=True, exist_ok=True)
ph.write_parquet(PHENO_OUT, compression="lz4", statistics=True)
print(f"✅ Saved updated pheno → {PHENO_OUT}")


[INFO] Loaded pheno: shape=(595, 10)
✅ Renamed: {'geo_accession': 'id_tissue', 'label_3class': 'label'}
[CHECK] Columns now:
id_tissue, sample_name, source_name, sample_type_raw, tissue_type_raw, age_at_surgery, race, sex, idat_basename, label
✅ Saved updated pheno → /kaggle/working/pheno_GSE225845_idtissue_lz4.parquet


In [19]:
# CHECK ALL WELL DONE -pheno
PHENO_PATH = "/kaggle/working/pheno_GSE225845_idtissue_lz4.parquet"

# --- Load pheno ---
ph = pl.read_parquet(PHENO_PATH)

# --- Summary ---
print(f"[INFO] Loaded pheno from: {PHENO_PATH}")
print(f"[INFO] Shape: {ph.shape}")

print("\n[INFO] Columns:")
for c in ph.columns:
    print(f"  - {c}")

[INFO] Loaded pheno from: /kaggle/working/pheno_GSE225845_idtissue_lz4.parquet
[INFO] Shape: (595, 10)

[INFO] Columns:
  - id_tissue
  - sample_name
  - source_name
  - sample_type_raw
  - tissue_type_raw
  - age_at_surgery
  - race
  - sex
  - idat_basename
  - label


In [ ]:
# CORRECTION ON DATA 225845
# Paths
BETA_IN   = "/kaggle/input/gse225845-parquet/GSE225845.parquet"
PHENO_IN  = "/kaggle/working/pheno_GSE225845_idtissue_lz4.parquet"
BETA_OUT  = "/kaggle/working/GSE225845_beta_with_idtissue_lz4.parquet"

# Load pheno (small) and prepare minimal mapping
ph = pl.read_parquet(PHENO_IN)
required = {"idat_basename", "id_tissue", "label"}
missing = required - set(ph.columns)
if missing:
    raise ValueError(f"Missing columns in pheno: {missing}")

# Deduplicate idat_basename if needed (keep first)
dup_count = ph.select(pl.col("idat_basename")).n_unique() != ph.height
if dup_count:
    print("⚠️  Duplicated 'idat_basename' detected in pheno; keeping first occurrence.")
    ph = ph.unique(subset=["idat_basename"], keep="first")

ph_min = ph.select([
    "idat_basename",
    "id_tissue",
    pl.col("label").alias("label_pheno")
])

# Lazy scan β
beta_lf = pl.scan_parquet(BETA_IN)

# Sanity checks on β header
beta_cols = set(beta_lf.columns)
for need in ["basenames", "label"]:
    if need not in beta_cols:
        raise ValueError(f"Column '{need}' not found in β matrix.")

# Join: basenames (β) ↔ idat_basename (pheno)
joined_lf = (
    beta_lf.join(
        ph_min.lazy(),
        left_on="basenames",
        right_on="idat_basename",
        how="left"
    )
)

# Consistency checks
# 1) Missing id_tissue after join
missing_id_tissue = joined_lf.select(pl.col("id_tissue").is_null().sum().alias("n_missing_id_tissue")).collect().item()
print(f"[CHECK] rows with missing id_tissue after join: {missing_id_tissue}")

# 2) Label mismatch (ignore nulls)
mismatch_expr = (
    pl.when(pl.col("label").is_null() | pl.col("label_pheno").is_null())
      .then(False)
      .otherwise(pl.col("label") != pl.col("label_pheno"))
)
mismatch_count = joined_lf.select(mismatch_expr.sum().alias("n_label_mismatch")).collect().item()
print(f"[CHECK] label mismatches (β vs pheno): {mismatch_count}")

# Show a few mismatches if any
if mismatch_count:
    ex = (
        joined_lf
        .select(["basenames", "idat_basename", "id_tissue", "label", "label_pheno"])
        .filter(mismatch_expr)
        .head(5)
        .collect()
    )
    print("[EXAMPLE] first 5 label mismatches:")
    print(ex)

# Drop helper columns & move id_tissue to the front
# I drop: accession_num, basenames (from β), idat_basename and label_pheno (from pheno)
to_drop = [c for c in ["accession_num", "basenames", "idat_basename", "label_pheno"] if c in joined_lf.columns]
clean_lf = joined_lf.drop(to_drop)

# Put id_tissue in front; keep the rest as-is (no heavy reordering)
final_lf = clean_lf.select(["id_tissue", pl.all().exclude("id_tissue")])

# Save (streaming if available)
Path(BETA_OUT).parent.mkdir(parents=True, exist_ok=True)
try:
    # Stream directly to Parquet if supported by your Polars version
    final_lf.sink_parquet(BETA_OUT, compression="lz4", statistics=True)
    print(f"✅ Wrote labeled β (streaming) → {BETA_OUT}")
except Exception:
    # Fallback: collect then write
    final_df = final_lf.collect(streaming=True)
    final_df.write_parquet(BETA_OUT, compression="lz4", statistics=True)
    print(f"✅ Wrote labeled β → {BETA_OUT}")

# Quick header preview
hdr = pl.read_parquet(BETA_OUT, n_rows=0)
print(f"[INFO] Output shape (rows, cols): {hdr.shape}")
print(f"[INFO] First 8 columns: {hdr.columns[:8]}")


/tmp/ipykernel_48/2767563486.py:35: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  beta_cols = set(beta_lf.columns)


[CHECK] rows with missing id_tissue after join: 0
[CHECK] label mismatches (β vs pheno): 0


/tmp/ipykernel_48/2767563486.py:78: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  to_drop = [c for c in ["accession_num", "basenames", "idat_basename", "label_pheno"] if c in joined_lf.columns]


✅ Wrote labeled β (streaming) → /kaggle/working/GSE225845_beta_with_idtissue_lz4.parquet
[INFO] Output shape (rows, cols): (0, 750428)
[INFO] First 8 columns: ['id_tissue', 'label', 'cg18478105', 'cg09835024', 'cg14361672', 'cg01763666', 'cg12950382', 'cg02115394']


In [22]:
# CHECK FAIL
ORIGINAL_BETA = "/kaggle/input/gse225845-parquet/GSE225845.parquet"
OUTPUT_BETA   = "/kaggle/working/GSE225845_beta_with_idtissue_lz4.parquet"

def parquet_shape(path: str) -> tuple[int, int]:
    lf = pl.scan_parquet(path)
    n_rows = lf.select(pl.len()).collect().item()
    n_cols = len(lf.collect_schema().names())
    return n_rows, n_cols

orig_shape = parquet_shape(ORIGINAL_BETA)
out_shape  = parquet_shape(OUTPUT_BETA)

print(f"[ORIGINAL] {ORIGINAL_BETA}")
print(f"  -> shape (rows, cols): {orig_shape}")

print(f"\n[OUTPUT]   {OUTPUT_BETA}")
print(f"  -> shape (rows, cols): {out_shape}")


[ORIGINAL] /kaggle/input/gse225845-parquet/GSE225845.parquet
  -> shape (rows, cols): (337, 750429)

[OUTPUT]   /kaggle/working/GSE225845_beta_with_idtissue_lz4.parquet
  -> shape (rows, cols): (337, 750428)


In [1]:
# BEGIN FROM TXT AGAIN -> norm
# 1) Paths
print("1) Paths — Start")
INPUT_TXT = "/kaggle/input/gse225845-normal-normalized-betas-txt/GSE225845_normal_normalized_betas.txt"
OUT_PAR   = "/kaggle/working/GSE225845_normal.parquet"
assert os.path.exists(INPUT_TXT), f"Input file not found: {INPUT_TXT}"
Path(OUT_PAR).parent.mkdir(parents=True, exist_ok=True)
print("1) Paths — End")

# 2) Columns to keep as strings
print("2) String columns — Start")
STRING_COLS = ["accession_num", "basenames"]
print("2) String columns — End")

# 3) Build lazy CSV scan (no full load)
print("3) scan_csv — Start")
lf = pl.scan_csv(
    INPUT_TXT,
    separator="\t",
    has_header=True,
    infer_schema_length=1000,   # light schema inference
    null_values=["NA", "NaN", ""]
)
print("3) scan_csv — End")

# --- Pre-write DIMENSION CHECKS on TXT (lazy/streaming) ---
print("3.1) TXT dimension checks — Start")
txt_cols = lf.collect_schema().names()
present_string_cols = [c for c in STRING_COLS if c in txt_cols]
missing_string_cols = [c for c in STRING_COLS if c not in txt_cols]
if missing_string_cols:
    print(f"⚠️  Missing expected string columns in TXT: {missing_string_cols}")

txt_ncols = len(txt_cols)
txt_nrows = lf.select(pl.len()).collect().item()
print(f"[TXT] shape (rows, cols): ({txt_nrows:,}, {txt_ncols:,})")
print(f"[TXT] present string cols: {present_string_cols}")
print("3.1) TXT dimension checks — End")

# 4) Cast dtypes lazily (strings stay Utf8, others to Float32)
print("4) dtype cast — Start")
exprs = []
for col in present_string_cols:
    exprs.append(pl.col(col).cast(pl.Utf8))
exprs.append(pl.all().exclude(present_string_cols).cast(pl.Float32))
lf = lf.with_columns(exprs)
print("4) dtype cast — End")

# 5) Write to Parquet (streaming sink)
print("5) Write Parquet — Start")
lf.sink_parquet(OUT_PAR, compression="lz4", statistics=True)
print(f"✅ Parquet saved to: {OUT_PAR}")
print("5) Write Parquet — End")

# 6) Post-write DIMENSION CHECKS on Parquet (lazy/streaming)
print("6) Parquet dimension checks — Start")
par_lf = pl.scan_parquet(OUT_PAR)
par_cols = par_lf.collect_schema().names()
par_ncols = len(par_cols)
par_nrows = par_lf.select(pl.len()).collect().item()
print(f"[PARQUET] shape (rows, cols): ({par_nrows:,}, {par_ncols:,})")

# Compare TXT vs Parquet dims
delta_rows = par_nrows - txt_nrows
delta_cols = par_ncols - txt_ncols
print(f"[COMPARE] rows delta (parquet - txt): {delta_rows:+}")
print(f"[COMPARE] cols delta (parquet - txt): {delta_cols:+}")

# Peek only string columns (cheap)
if present_string_cols:
    peek = (
        par_lf.select(present_string_cols)
              .head(5)
              .collect()
    )
    print("[PARQUET] first 5 rows (string columns):")
    print(peek)
else:
    print("ℹ️  No string columns present to preview.")
print("6) Parquet dimension checks — End")


1) Paths — Start
1) Paths — End
2) String columns — Start
2) String columns — End
3) scan_csv — Start
3) scan_csv — End
3.1) TXT dimension checks — Start
[TXT] shape (rows, cols): (113, 750,428)
[TXT] present string cols: ['accession_num', 'basenames']
3.1) TXT dimension checks — End
4) dtype cast — Start
4) dtype cast — End
5) Write Parquet — Start


/tmp/ipykernel_48/3185454868.py:56: UserWarning: encountered expression deeper than 512 elements; this may overflow the stack, consider refactoring
  lf.sink_parquet(OUT_PAR, compression="lz4", statistics=True)


✅ Parquet saved to: /kaggle/working/GSE225845_normal.parquet
5) Write Parquet — End
6) Parquet dimension checks — Start
[PARQUET] shape (rows, cols): (113, 750,428)
[COMPARE] rows delta (parquet - txt): +0
[COMPARE] cols delta (parquet - txt): +0
[PARQUET] first 5 rows (string columns):
shape: (5, 2)
┌───────────────┬─────────────────────┐
│ accession_num ┆ basenames           │
│ ---           ┆ ---                 │
│ str           ┆ str                 │
╞═══════════════╪═════════════════════╡
│ 12598         ┆ 203723060067_R04C01 │
│ 12919         ┆ 203723060067_R08C01 │
│ 12489         ┆ 204425700003_R01C01 │
│ 12367         ┆ 204425700003_R02C01 │
│ 12616         ┆ 204425700003_R03C01 │
└───────────────┴─────────────────────┘
6) Parquet dimension checks — End


In [ ]:
# BEGIN FROM TXT AGAIN -> adj
# 1) Paths
print("1) Paths — Start")
INPUT_TXT = "/kaggle/input/gse225845-adjnorm-normalized-betas-txt/GSE225845_adjnorm_normalized_betas.txt"
OUT_PAR   = "/kaggle/working/GSE225845_adj.parquet"
assert os.path.exists(INPUT_TXT), f"Input file not found: {INPUT_TXT}"
Path(OUT_PAR).parent.mkdir(parents=True, exist_ok=True)
print("1) Paths — End")

# 2) Columns to keep as strings
print("2) String columns — Start")
STRING_COLS = ["accession_num", "basenames"]
print("2) String columns — End")

# 3) Build lazy CSV scan (no full load)
print("3) scan_csv — Start")
lf = pl.scan_csv(
    INPUT_TXT,
    separator="\t",
    has_header=True,
    infer_schema_length=1000,   # light schema inference
    null_values=["NA", "NaN", ""]
)
print("3) scan_csv — End")

# --- Pre-write DIMENSION CHECKS on TXT (lazy/streaming) ---
print("3.1) TXT dimension checks — Start")
txt_cols = lf.collect_schema().names()
present_string_cols = [c for c in STRING_COLS if c in txt_cols]
missing_string_cols = [c for c in STRING_COLS if c not in txt_cols]
if missing_string_cols:
    print(f"⚠️  Missing expected string columns in TXT: {missing_string_cols}")

txt_ncols = len(txt_cols)
txt_nrows = lf.select(pl.len()).collect().item()
print(f"[TXT] shape (rows, cols): ({txt_nrows:,}, {txt_ncols:,})")
print(f"[TXT] present string cols: {present_string_cols}")
print("3.1) TXT dimension checks — End")

# 4) Cast dtypes lazily (strings stay Utf8, others to Float32)
print("4) dtype cast — Start")
exprs = []
for col in present_string_cols:
    exprs.append(pl.col(col).cast(pl.Utf8))
exprs.append(pl.all().exclude(present_string_cols).cast(pl.Float32))
lf = lf.with_columns(exprs)
print("4) dtype cast — End")

# 5) Write to Parquet (streaming sink)
print("5) Write Parquet — Start")
lf.sink_parquet(OUT_PAR, compression="lz4", statistics=True)
print(f"✅ Parquet saved to: {OUT_PAR}")
print("5) Write Parquet — End")

# 6) Post-write DIMENSION CHECKS on Parquet (lazy/streaming)
print("6) Parquet dimension checks — Start")
par_lf = pl.scan_parquet(OUT_PAR)
par_cols = par_lf.collect_schema().names()
par_ncols = len(par_cols)
par_nrows = par_lf.select(pl.len()).collect().item()
print(f"[PARQUET] shape (rows, cols): ({par_nrows:,}, {par_ncols:,})")

# Compare TXT vs Parquet dims
delta_rows = par_nrows - txt_nrows
delta_cols = par_ncols - txt_ncols
print(f"[COMPARE] rows delta (parquet - txt): {delta_rows:+}")
print(f"[COMPARE] cols delta (parquet - txt): {delta_cols:+}")

# Peek only string columns (cheap)
if present_string_cols:
    peek = (
        par_lf.select(present_string_cols)
              .head(5)
              .collect()
    )
    print("[PARQUET] first 5 rows (string columns):")
    print(peek)
else:
    print("ℹ️  No string columns present to preview.")
print("6) Parquet dimension checks — End")


1) Paths — Start
1) Paths — End
2) String columns — Start
2) String columns — End
3) scan_csv — Start
3) scan_csv — End
3.1) TXT dimension checks — Start
[TXT] shape (rows, cols): (140, 750,428)
[TXT] present string cols: ['accession_num', 'basenames']
3.1) TXT dimension checks — End
4) dtype cast — Start
4) dtype cast — End
5) Write Parquet — Start


/tmp/ipykernel_48/2076944994.py:56: UserWarning: encountered expression deeper than 512 elements; this may overflow the stack, consider refactoring
  lf.sink_parquet(OUT_PAR, compression="lz4", statistics=True)


In [1]:
# BEGIN FROM TXT AGAIN -> tum
# 1) Paths
print("1) Paths — Start")
INPUT_TXT = "/kaggle/input/gse225845-adjnorm-normalized-betas-txt/GSE225845_adjnorm_normalized_betas.txt"
OUT_PAR   = "/kaggle/working/GSE225845_adj.parquet"
assert os.path.exists(INPUT_TXT), f"Input file not found: {INPUT_TXT}"
Path(OUT_PAR).parent.mkdir(parents=True, exist_ok=True)
print("1) Paths — End")

# 2) Columns to keep as strings 
print("2) String columns — Start")
STRING_COLS = ["accession_num", "basenames"]  
print("2) String columns — End")

# 3) Build lazy CSV scan (no full load)
print("3) scan_csv — Start")
lf = pl.scan_csv(
    INPUT_TXT,
    separator="\t",
    has_header=True,
    infer_schema_length=1000,
    null_values=["NA", "NaN", ""],
    ignore_errors=True,       
)
print("3) scan_csv — End")

# --- Pre-write LIGHT checks ---
print("3.1) TXT schema check — Start")
txt_cols = lf.collect_schema().names()
present_string_cols = [c for c in STRING_COLS if c in txt_cols]
missing_string_cols = [c for c in STRING_COLS if c not in txt_cols]
if missing_string_cols:
    print(f"⚠️  Missing expected string columns in TXT: {missing_string_cols}")
print(f"[TXT] ncols: {len(txt_cols):,}")
print(f"[TXT] present string cols: {present_string_cols}")
print("3.1) TXT schema check — End")

# 4) Cast dtypes lazily
print("4) dtype cast — Start")
num_candidates = [c for c in txt_cols if c not in present_string_cols]
exprs = []
if present_string_cols:
    exprs += [pl.col(c).cast(pl.Utf8) for c in present_string_cols]
if num_candidates:
    exprs += [pl.col(num_candidates).cast(pl.Float32)]
lf = lf.with_columns(exprs)
print("4) dtype cast — End")

# 5) Write to Parquet (streaming sink) 
print("5) Write Parquet — Start")
lf.sink_parquet(
    OUT_PAR,
    compression="lz4",
    statistics=False,     
    row_group_size=10_000 
)
print(f"✅ Parquet saved to: {OUT_PAR}")
print("5) Write Parquet — End")

# 6) Post-write DIMENSION CHECKS
print("6) Parquet dimension checks — Start")
par_lf = pl.scan_parquet(OUT_PAR)
par_cols = par_lf.collect_schema().names()
par_ncols = len(par_cols)
print(f"[PARQUET] ncols: {par_ncols:,}")

peek_cols = [c for c in present_string_cols if c in par_cols]
if peek_cols:
    peek = par_lf.select(peek_cols).head(5).collect()
    print("[PARQUET] first 5 rows (string columns):")
    print(peek)
else:
    print("ℹ️  No string columns present to preview.")
print("6) Parquet dimension checks — End")


1) Paths — Start
1) Paths — End
2) String columns — Start
2) String columns — End
3) scan_csv — Start
3) scan_csv — End
3.1) TXT schema check — Start
[TXT] ncols: 750,428
[TXT] present string cols: ['accession_num', 'basenames']
3.1) TXT schema check — End
4) dtype cast — Start
4) dtype cast — End
5) Write Parquet — Start


/tmp/ipykernel_109/4157372863.py:57: UserWarning: encountered expression deeper than 512 elements; this may overflow the stack, consider refactoring
  lf.sink_parquet(


✅ Parquet saved to: /kaggle/working/GSE225845_adj.parquet
5) Write Parquet — End
6) Parquet dimension checks — Start
[PARQUET] ncols: 750,428
[PARQUET] first 5 rows (string columns):
shape: (5, 2)
┌───────────────┬─────────────────────┐
│ accession_num ┆ basenames           │
│ ---           ┆ ---                 │
│ str           ┆ str                 │
╞═══════════════╪═════════════════════╡
│ 24231         ┆ 203723050007_R01C01 │
│ 24471         ┆ 203723050007_R03C01 │
│ 24508         ┆ 203723050007_R06C01 │
│ 23891         ┆ 203723050007_R07C01 │
│ 27101         ┆ 203723050008_R01C01 │
└───────────────┴─────────────────────┘
6) Parquet dimension checks — End


In [2]:
# POST-WRITE CHECK — GSE225845_adj.parquet
PAR_FILE = Path("/kaggle/working/GSE225845_adj.parquet")
assert PAR_FILE.exists(), f"❌ File not found: {PAR_FILE}"

print("[INFO] Checking Parquet:", PAR_FILE)

# 1) Lazy load to inspect schema (no full read)
lf = pl.scan_parquet(PAR_FILE)

# 2) Shape estimation (streaming)
n_rows = lf.select(pl.len()).collect().item()
n_cols = len(lf.collect_schema().names())
print(f"[OK] Shape: {n_rows:,} rows × {n_cols:,} columns")


[INFO] Checking Parquet: /kaggle/working/GSE225845_adj.parquet
[OK] Shape: 140 rows × 750,428 columns


In [ ]:
# TXT (TUMORS) -> PARQUET
# ----------- CONFIG ------------------------------------------------
INPUT_PATH     = "/kaggle/input/gse225845-tumors-normalized-betas-txt/GSE225845_tumors_normalized_betas.txt"
OUTPUT_FILE    = "/kaggle/working/GSE225845_tum.parquet"

META_COLS      = ["accession_num", "basenames"]  # metadata cols as in TXT
FLOAT_DTYPE    = np.float32
NA_STRINGS     = ["NA", "NaN", "nan", "", "null", "NULL"]
ROW_BLOCK_SIZE = 1024
NUM_SAMPLES    = 224   # expected number of rows

# ----------- PREP --------------------------------------------------
def ensure_parent(path: str):
    Path(path).parent.mkdir(parents=True, exist_ok=True)

assert os.path.exists(INPUT_PATH), f"Input file not found: {INPUT_PATH}"
ensure_parent(OUTPUT_FILE)
print("STEP 1 - imports and config ready")

# ----------- HEADER DISCOVERY (light) ------------------------------
print("STEP 2 - reading header with csv module")
all_cols: List[str] = []
try:
    with open(INPUT_PATH, 'r', newline='', encoding='utf-8') as f:
        reader = csv.reader(f, delimiter='\t')
        all_cols = next(reader)
except Exception as e:
    # Fallback very light
    hdr = pd.read_csv(INPUT_PATH, sep="\t", nrows=0)
    all_cols = hdr.columns.tolist()

meta_cols = [c for c in META_COLS if c in all_cols]
cpg_cols  = [c for c in all_cols if c not in meta_cols]
print(f"  Total columns: {len(all_cols):,}")
print(f"  Metadata columns detected: {meta_cols}")
print(f"  CpG columns: {len(cpg_cols):,}")

# ----------- OPTIONAL tiny peek of TXT meta only -------------------
try:
    peek_meta = pd.read_csv(INPUT_PATH, sep="\t", nrows=3, usecols=meta_cols)
    print("[PEEK TXT] first 3 rows (metadata only):")
    print(peek_meta)
except Exception as e:
    print(f"[PEEK TXT] skipped (reason: {e})")

print(f"[INFO] expected #samples (rows) = {NUM_SAMPLES:,}")
print(f"[INFO] #CpGs (cols)             = {len(cpg_cols):,}")

# ----------- SCHEMA (Arrow + Pandas dtypes) -----------------------
print("STEP 3 - Building Arrow schema and Pandas dtype map")
dtype_map: Dict[str, any] = {c: "string" for c in meta_cols}
dtype_map.update({c: FLOAT_DTYPE for c in cpg_cols})

pa_fields = [pa.field(c, pa.string())   for c in meta_cols] + \
            [pa.field(c, pa.float32())  for c in cpg_cols]
pa_schema = pa.schema(pa_fields)
print("  Schema ready.")

# ----------- ParquetWriter (SAFE settings) ------------------------
print("STEP 4 - Initializing ParquetWriter")
writer = pq.ParquetWriter(
    OUTPUT_FILE,
    pa_schema,
    compression="lz4",
    use_dictionary=False,
    data_page_size=1 << 20,   # 1MB
    write_statistics=False,   # disable stats to reduce RAM
)
print(f"  Writer initialized: {OUTPUT_FILE}")

# ----------- STREAMING WRITE --------------------------------------
print("STEP 5 - Streaming TXT -> Parquet")
reader = pd.read_csv(
    INPUT_PATH,
    sep="\t",
    skiprows=[0],              
    names=all_cols,
    dtype=dtype_map,
    chunksize=ROW_BLOCK_SIZE,
    na_values=NA_STRINGS,
    keep_default_na=True,
    low_memory=False,
)

written_rows = 0
chunk_idx = 0

try:
    for chunk in reader:
        chunk_idx += 1
        n_rows_chunk = len(chunk)
        table_chunk = pa.Table.from_pandas(chunk, schema=pa_schema, preserve_index=False)
        writer.write_table(table_chunk)
        written_rows += n_rows_chunk

        # tiny housekeeping
        del chunk, table_chunk
        if (chunk_idx % 5) == 0:
            gc.collect()
            print(f"  - processed chunk {chunk_idx}, rows written: {written_rows:,}/{NUM_SAMPLES:,}")
finally:
    writer.close()
    del reader
    gc.collect()

print("STEP 6 - Finished writing chunks, writer closed.")
assert written_rows == NUM_SAMPLES, f"Written rows {written_rows} != expected {NUM_SAMPLES}"
print(f"[DONE] Wrote single Parquet -> {OUTPUT_FILE}")


In [6]:
# POST-WRITE CHECK — GSE225845_adj.parquet
PAR_FILE = Path("/kaggle/working/GSE225845_tum.parquet")
assert PAR_FILE.exists(), f"❌ File not found: {PAR_FILE}"

print("[INFO] Checking Parquet:", PAR_FILE)

# 1) Lazy load to inspect schema (no full read)
lf = pl.scan_parquet(PAR_FILE)

# 2) Shape estimation (streaming)
n_rows = lf.select(pl.len()).collect().item()
n_cols = len(lf.collect_schema().names())
print(f"[OK] Shape: {n_rows:,} rows × {n_cols:,} columns")


[INFO] Checking Parquet: /kaggle/working/GSE225845_tum.parquet
[OK] Shape: 224 rows × 750,428 columns


In [9]:
# PARQUET MERGER: CONCATENATES MULTIPLE PARQUET FILES INTO A SINGLE FILE
# Files are concatenated vertically (adding rows/samples).
# Adds a 'label' column to identify the sample type.
# List of tuples: (Parquet file path, label value)
FILES_WITH_LABELS: List[Tuple[str, int]] = [
    ("/kaggle/working/GSE225845_normal.parquet", 0),  # 0 = normal
    ("/kaggle/working/GSE225845_adj.parquet", 1),       # 1 = normal-adjacent
    ("/kaggle/working/GSE225845_tum.parquet", 2),  # 2 = breast cancer
]

OUTPUT_PATH = "/kaggle/working/GSE225845_all_samples_with_labels.parquet"

# Columns to keep as strings (accession_num, basenames)
# Ensure 'label' is the first column after metadata for convention.
METADATA_COLS_TO_KEEP = ["accession_num", "basenames"] 
LABEL_COL_NAME = "label"

# Build lazy frames with labels
print("STEP 1: Building Lazy Frames and adding 'label' column...")

lfs = []
for path, lab in FILES_WITH_LABELS:
    # 1. Check if the file exists before proceeding
    if not Path(path).exists():
        print(f"ATTENTION: File not found and skipped: {path}")
        continue
    
    # 2. Read the file in lazy mode
    lf = (
        pl.scan_parquet(path)  # lazy, no full load in RAM
        .with_columns(
            pl.lit(lab)
              .cast(pl.Int8)    # small integer, saves space
              .alias(LABEL_COL_NAME)
        )
    )
    
    # 3. Reorganize columns: metadata + label + CpG
    # This reorganization ensures the schema is uniform before concatenation.
    col_order = [pl.col(c) for c in METADATA_COLS_TO_KEEP]
    col_order.append(pl.col(LABEL_COL_NAME))
    col_order.append(pl.exclude(METADATA_COLS_TO_KEEP + [LABEL_COL_NAME]))
    
    lf = lf.select(*col_order)
    
    lfs.append(lf)
    print(f"  - Added {path} (Label: {lab})")

if not lfs:
    raise FileNotFoundError("FATAL: No valid Parquet file was found. Cannot concatenate.")
    
# Vertical concatenation: same columns, different samples
print("\nSTEP 2: Concatenating all Lazy Frames (Vertical merge)...")

# Polars handles vertical concatenation of frames with identical schemas.
lf_all = pl.concat(lfs, how="vertical")

print("  Concatenation successful (Lazy mode).")

# Optional quick sanity check (small sample)
print("\nSTEP 3: Performing a quick data summary check...")
try:
    summary_df = (
        lf_all
        .select(
            pl.len().alias("n_samples_total"),
            pl.col(LABEL_COL_NAME).value_counts().sort(LABEL_COL_NAME)
        )
        .collect()
    )
    print(summary_df)
except Exception as e:
    print(f"ATTENTION: Could not perform sanity check (perhaps due to schema): {e}")

# Write single Parquet in streaming mode
print(f"\nSTEP 4: Writing single combined Parquet file to {OUTPUT_PATH}...")

# The use of sink_parquet ensures optimized and low-RAM writing
lf_all.sink_parquet(
    OUTPUT_PATH,
    compression="lz4",
    statistics=True,
    # The row_group_size parameter can be useful for subsequent analysis, but we use Polars' default
)

print(f"\n✅ Written combined Parquet with labels -> {OUTPUT_PATH}")
print("Process completed.")


STEP 1: Building Lazy Frames and adding 'label' column...
  - Added /kaggle/working/GSE225845_normal.parquet (Label: 0)
  - Added /kaggle/working/GSE225845_adj.parquet (Label: 1)
  - Added /kaggle/working/GSE225845_tum.parquet (Label: 2)

STEP 2: Concatenating all Lazy Frames (Vertical merge)...
  Concatenation successful (Lazy mode).

STEP 3: Performing a quick data summary check...
ATTENTION: Could not perform sanity check (perhaps due to schema): Expr.sort() takes 1 positional argument but 2 were given

STEP 4: Writing single combined Parquet file to /kaggle/working/GSE225845_all_samples_with_labels.parquet...

✅ Written combined Parquet with labels -> /kaggle/working/GSE225845_all_samples_with_labels.parquet
Process completed.


In [10]:
# POST-WRITE CHECK — GSE225845_adj.parquet
PAR_FILE = Path("/kaggle/working/GSE225845_all_samples_with_labels.parquet")
assert PAR_FILE.exists(), f"❌ File not found: {PAR_FILE}"

print("[INFO] Checking Parquet:", PAR_FILE)

# 1) Lazy load to inspect schema (no full read)
lf = pl.scan_parquet(PAR_FILE)

# 2) Shape estimation (streaming)
n_rows = lf.select(pl.len()).collect().item()
n_cols = len(lf.collect_schema().names())
print(f"[OK] Shape: {n_rows:,} rows × {n_cols:,} columns")


[INFO] Checking Parquet: /kaggle/working/GSE225845_all_samples_with_labels.parquet
[OK] Shape: 477 rows × 750,429 columns


In [11]:
# CORRECTION ON DATA 225845
# ---------- Paths ----------
BETA_IN   = "/kaggle/working/GSE225845_all_samples_with_labels.parquet"
# Use the pheno you just saved with renamed columns:
PHENO_IN  = "/kaggle/input/pheno-gse225845/pheno_GSE225845.parquet"
BETA_OUT  = "/kaggle/working/GSE225845.parquet"

# ---------- Load pheno (small) and prepare minimal mapping ----------
ph = pl.read_parquet(PHENO_IN)
required = {"idat_basename", "id_tissue", "label"}
missing = required - set(ph.columns)
if missing:
    raise ValueError(f"Missing columns in pheno: {missing}")

# Deduplicate idat_basename if needed (keep first)
dup_count = ph.select(pl.col("idat_basename")).n_unique() != ph.height
if dup_count:
    print("⚠️  Duplicated 'idat_basename' detected in pheno; keeping first occurrence.")
    ph = ph.unique(subset=["idat_basename"], keep="first")

ph_min = ph.select([
    "idat_basename",
    "id_tissue",
    pl.col("label").alias("label_pheno")
])

# ---------- Lazy scan β ----------
beta_lf = pl.scan_parquet(BETA_IN)

# Sanity checks on β header
beta_cols = set(beta_lf.columns)
for need in ["basenames", "label"]:
    if need not in beta_cols:
        raise ValueError(f"Column '{need}' not found in β matrix.")

# ---------- Join: basenames (β) ↔ idat_basename (pheno) ----------
joined_lf = (
    beta_lf.join(
        ph_min.lazy(),
        left_on="basenames",
        right_on="idat_basename",
        how="left"
    )
)

# ---------- Consistency checks ----------
# 1) Missing id_tissue after join
missing_id_tissue = joined_lf.select(pl.col("id_tissue").is_null().sum().alias("n_missing_id_tissue")).collect().item()
print(f"[CHECK] rows with missing id_tissue after join: {missing_id_tissue}")

# 2) Label mismatch (ignore nulls)
mismatch_expr = (
    pl.when(pl.col("label").is_null() | pl.col("label_pheno").is_null())
      .then(False)
      .otherwise(pl.col("label") != pl.col("label_pheno"))
)
mismatch_count = joined_lf.select(mismatch_expr.sum().alias("n_label_mismatch")).collect().item()
print(f"[CHECK] label mismatches (β vs pheno): {mismatch_count}")

# Show a few mismatches if any
if mismatch_count:
    ex = (
        joined_lf
        .select(["basenames", "idat_basename", "id_tissue", "label", "label_pheno"])
        .filter(mismatch_expr)
        .head(5)
        .collect()
    )
    print("[EXAMPLE] first 5 label mismatches:")
    print(ex)

# ---------- Drop helper columns & move id_tissue to the front ----------
# I drop: accession_num, basenames (from β), idat_basename and label_pheno (from pheno)
to_drop = [c for c in ["accession_num", "basenames", "idat_basename", "label_pheno"] if c in joined_lf.columns]
clean_lf = joined_lf.drop(to_drop)

# Put id_tissue in front; keep the rest as-is (no heavy reordering)
final_lf = clean_lf.select(["id_tissue", pl.all().exclude("id_tissue")])

# ---------- Save (streaming if available) ----------
Path(BETA_OUT).parent.mkdir(parents=True, exist_ok=True)
try:
    # Stream directly to Parquet if supported by your Polars version
    final_lf.sink_parquet(BETA_OUT, compression="lz4", statistics=True)
    print(f"✅ Wrote labeled β (streaming) → {BETA_OUT}")
except Exception:
    # Fallback: collect then write
    final_df = final_lf.collect(streaming=True)
    final_df.write_parquet(BETA_OUT, compression="lz4", statistics=True)
    print(f"✅ Wrote labeled β → {BETA_OUT}")

# ---------- Quick header preview ----------
hdr = pl.read_parquet(BETA_OUT, n_rows=0)
print(f"[INFO] Output shape (rows, cols): {hdr.shape}")
print(f"[INFO] First 8 columns: {hdr.columns[:8]}")


/tmp/ipykernel_109/2019692995.py:35: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  beta_cols = set(beta_lf.columns)


[CHECK] rows with missing id_tissue after join: 0
[CHECK] label mismatches (β vs pheno): 0


/tmp/ipykernel_109/2019692995.py:78: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  to_drop = [c for c in ["accession_num", "basenames", "idat_basename", "label_pheno"] if c in joined_lf.columns]


✅ Wrote labeled β (streaming) → /kaggle/working/GSE225845.parquet
[INFO] Output shape (rows, cols): (0, 750428)
[INFO] First 8 columns: ['id_tissue', 'label', 'cg18478105', 'cg09835024', 'cg14361672', 'cg01763666', 'cg12950382', 'cg02115394']


In [12]:
# CHECK 
ORIGINAL_BETA = "/kaggle/working/GSE225845.parquet"
OUTPUT_BETA   = "/kaggle/working/GSE225845.parquet"

def parquet_shape(path: str) -> tuple[int, int]:
    lf = pl.scan_parquet(path)
    n_rows = lf.select(pl.len()).collect().item()
    n_cols = len(lf.collect_schema().names())
    return n_rows, n_cols

orig_shape = parquet_shape(ORIGINAL_BETA)
out_shape  = parquet_shape(OUTPUT_BETA)

print(f"[ORIGINAL] {ORIGINAL_BETA}")
print(f"  -> shape (rows, cols): {orig_shape}")

print(f"\n[OUTPUT]   {OUTPUT_BETA}")
print(f"  -> shape (rows, cols): {out_shape}")


[ORIGINAL] /kaggle/working/GSE225845.parquet
  -> shape (rows, cols): (477, 750428)

[OUTPUT]   /kaggle/working/GSE225845.parquet
  -> shape (rows, cols): (477, 750428)


In [14]:
# Align pheno with β-matrix (keep only samples present in β)
BETA_PATH  = "/kaggle/working/GSE225845.parquet"   
PHENO_PATH = "/kaggle/input/pheno-gse225845/pheno_GSE225845.parquet"
OUT_PHENO  = "/kaggle/working/pheno_GSE225845_filtered.parquet"

# 1) Load (lazy -> collect only keys)
beta_lf = pl.scan_parquet(BETA_PATH).select("id_tissue")
pheno_lf = pl.scan_parquet(PHENO_PATH)

# 2) Inner join on sample_id → keep only samples with methylation data
aligned_pheno = (
    pheno_lf.join(beta_lf, on="id_tissue", how="inner")
    .collect()
)

# 3) Save filtered pheno
aligned_pheno.write_parquet(OUT_PHENO, compression="lz4")

print(f"[OK] Saved filtered pheno → {OUT_PHENO}")
print(f"Original pheno samples: {pheno_lf.select(pl.len()).collect().item()}")
print(f"Filtered pheno samples: {aligned_pheno.height}")


[OK] Saved filtered pheno → /kaggle/working/pheno_GSE225845_filtered.parquet
Original pheno samples: 595
Filtered pheno samples: 477


In [16]:
# CHECK BASE-NAME CONSISTENCY BETWEEN BETA AND PHENO TABLES
# --- Paths ---------------------------------------------------
BETA_PATH  = "/kaggle/working/GSE225845_all_samples_with_labels.parquet"
PHENO_PATH = "/kaggle/input/pheno-gse225845/pheno_GSE225845.parquet"

# --- Load minimal columns (lazy mode for memory safety) -----
beta_lf  = pl.scan_parquet(BETA_PATH).select("basenames")
pheno_lf = pl.scan_parquet(PHENO_PATH).select("idat_basename")

# --- Collect as sets ----------------------------------------
beta_basenames  = set(beta_lf.collect().to_series().drop_nulls().to_list())
pheno_basenames = set(pheno_lf.collect().to_series().drop_nulls().to_list())

# --- Check unique counts ------------------------------------
print(f"[INFO] β basenames (unique): {len(beta_basenames):,}")
print(f"[INFO] pheno idat_basename (unique): {len(pheno_basenames):,}")

# --- Basenames in β not found in pheno ------------------
missing_in_pheno = beta_basenames - pheno_basenames
if missing_in_pheno:
    print(f"⚠️  {len(missing_in_pheno)} basenames in β not found in pheno.")
else:
    print("✅ All β basenames are found in pheno.")

# --- Basenames in pheno not found in β ------------------
missing_in_beta = pheno_basenames - beta_basenames
if missing_in_beta:
    print(f"⚠️  {len(missing_in_beta)} idat_basename in pheno not found in β.")
else:
    print("✅ All pheno idat_basename are found in β.")

# --- Optional: show small preview of mismatches --------------
if missing_in_pheno:
    print("\n[EXAMPLES missing in pheno]:")
    for x in list(missing_in_pheno)[:5]:
        print("  ", x)

if missing_in_beta:
    print("\n[EXAMPLES missing in β]:")
    for x in list(missing_in_beta)[:5]:
        print("  ", x)


[INFO] β basenames (unique): 477
[INFO] pheno idat_basename (unique): 595
✅ All β basenames are found in pheno.
⚠️  118 idat_basename in pheno not found in β.

[EXAMPLES missing in β]:
   205624900132_R05C01
   205624900118_R07C01
   205624900056_R08C01
   205624900118_R06C01
   205624900118_R01C01


In [19]:
# ONE-TO-MANY CHECK: beta.<basename> ↔ pheno.<idat_basename>
# Auto-detect columns, normalize keys, report ambiguous matches.
BETA_PATH  = "/kaggle/working/GSE225845_all_samples_with_labels.parquet"
PHENO_PATH = "/kaggle/input/pheno-gse225845/pheno_GSE225845.parquet"
REPORT_CSV  = "/kaggle/working/basename_manymatch_report.csv"
NORMALIZE   = True

# --- Load minimal columns (lazy mode for memory safety) -----
beta_lf  = pl.scan_parquet(BETA_PATH).select("basenames")
pheno_lf = pl.scan_parquet(PHENO_PATH).select("idat_basename")

# --- Collect as sets ----------------------------------------
beta_basenames  = set(beta_lf.collect().to_series().drop_nulls().to_list())
pheno_basenames = set(pheno_lf.collect().to_series().drop_nulls().to_list())

# --- Check unique counts ------------------------------------
print(f"[INFO] β basenames (unique): {len(beta_basenames):,}")
print(f"[INFO] pheno idat_basename (unique): {len(pheno_basenames):,}")


# ---- Helpers ------------------------------------------------
def pick_first(existing_cols, candidates):
    for c in candidates:
        if c in existing_cols:
            return c
    return None

def norm_expr(colname: str):
    e = pl.col(colname)
    if NORMALIZE:
        e = e.cast(pl.Utf8).str.strip_chars().str.to_lowercase()
    return e

# ---- Detect columns ----------------------------------------
beta_cols  = pl.scan_parquet(BETA_PATH).collect_schema().names()
pheno_cols = pl.scan_parquet(PHENO_PATH).collect_schema().names()

BETA_COL  = pick_first(beta_cols,  ["basenames", "basename", "idat_basename", "file_basename"])
PHENO_COL = pick_first(pheno_cols, ["idat_basename", "basenames", "basename", "file_basename"])

# Optional ID column for nicer reporting
PHENO_ID  = pick_first(pheno_cols, ["sample_id","geo_accession","GEO_accession","gsm","GSM",
                                    "accession_num","sample","Sample_ID","sample_name"])

if BETA_COL is None or PHENO_COL is None:
    raise ValueError(f"Could not detect basename columns. "
                     f"Beta has: {beta_cols[:10]}..., Pheno has: {pheno_cols[:10]}...")

print(f"[INFO] Using beta column:  {BETA_COL}")
print(f"[INFO] Using pheno column: {PHENO_COL}")
print(f"[INFO] Using pheno ID:     {PHENO_ID if PHENO_ID else '(none)'}")

# ---- Build lazy frames with normalized keys ----------------
beta_keys = (
    pl.scan_parquet(BETA_PATH)
      .select(norm_expr(BETA_COL).alias("key"))
      .drop_nulls()
)
beta_unique = beta_keys.unique()

pheno_min = pl.scan_parquet(PHENO_PATH).select([
    norm_expr(PHENO_COL).alias("key"),
    pl.col(PHENO_COL).alias("pheno_raw_basename"),
    *( [pl.col(PHENO_ID).alias("pheno_id")] if PHENO_ID else [] )
]).drop_nulls()

# ---- Count matches in pheno for keys present in beta -------
pheno_counts = (
    pheno_min.join(beta_unique, on="key", how="inner")
             .group_by("key")
             .agg([
                 pl.len().alias("pheno_count"),
                 pl.col("pheno_raw_basename").n_unique().alias("distinct_raw_names"),
                 *( [pl.col("pheno_id").n_unique().alias("distinct_ids")] if PHENO_ID else [] ),
                 *( [pl.col("pheno_id").unique().alias("example_ids")] if PHENO_ID else [] ),
             ])
)

# ---- Duplicates inside beta itself -------------------------
beta_dups = (
    beta_keys.group_by("key").agg(pl.len().alias("beta_count"))
             .filter(pl.col("beta_count") > 1)
)

# ---- Consolidated report -----------------------------------
report = (
    pheno_counts.join(beta_dups, on="key", how="left")
                .with_columns(pl.col("beta_count").fill_null(1))
                .sort(["pheno_count","beta_count"], descending=True)
)

problematic = report.filter(pl.col("pheno_count") > 1)
ok_unique   = report.filter((pl.col("pheno_count") == 1) & (pl.col("beta_count") == 1))

prob_df = problematic.collect()
ok_df   = ok_unique.collect()
dup_beta_n = beta_dups.select(pl.len()).collect().item()

print("=== SUMMARY ===")
print(f"β unique basenames: {beta_unique.select(pl.len()).collect().item():,}")
print(f"Pheno many-matches (β basename → ≥2 pheno rows): {prob_df.height:,}")
print(f"β internal duplicates: {dup_beta_n:,}")

if prob_df.height > 0:
    show_cols = ["key","pheno_count","distinct_raw_names","beta_count"]
    if PHENO_ID: show_cols += ["distinct_ids","example_ids"]
    print("\nTop 10 problematic basenames:")
    print(prob_df.select(show_cols).head(10))
else:
    print("\n✅ No one-to-many matches: each β basename maps to exactly one pheno row.")

# ---- Save CSV with problematic mappings --------------------
if prob_df.height > 0:
    Path(REPORT_CSV).parent.mkdir(parents=True, exist_ok=True)
    out = prob_df
    if PHENO_ID:
        out = out.with_columns(pl.col("example_ids").list.slice(0, 10).alias("example_ids_first10")) \
                 .drop("example_ids")
    out.write_csv(REPORT_CSV)
    print(f"\n[REPORT] Saved: {REPORT_CSV}")
else:
    print("\n[REPORT] Not written (no problematic mappings).")


[INFO] β basenames (unique): 477
[INFO] pheno idat_basename (unique): 595
[INFO] Using beta column:  basenames
[INFO] Using pheno column: idat_basename
[INFO] Using pheno ID:     sample_name
=== SUMMARY ===
β unique basenames: 477
Pheno many-matches (β basename → ≥2 pheno rows): 0
β internal duplicates: 0

✅ No one-to-many matches: each β basename maps to exactly one pheno row.

[REPORT] Not written (no problematic mappings).


In [ ]:
# CHECK NO LABEL IN 287331
BETA_PATH = "/kaggle/input/gse287331-parquet/GSE287331.parquet"

print("Parquet dimension checks — Start")

# Schema via lazy scan (non carica tutto)
par_lf   = pl.scan_parquet(BETA_PATH)
schema   = par_lf.schema  # dict: {col_name: dtype}
par_cols = list(schema.keys())
par_ncols = len(par_cols)

print(f"[PARQUET] ncols: {par_ncols:,}")
print(f"[PARQUET] first 10 columns: {par_cols[:10]}")
print(f"[PARQUET] last  10 columns: {par_cols[-10:]}")

# Trova le colonne stringa (se fossero label, id, ecc.)
string_cols = [name for name, dtype in schema.items() if dtype == pl.Utf8]

if string_cols:
    print(f"⚠️  String columns present: {string_cols}")
    peek = par_lf.select(string_cols).head(5).collect()
    print("[PARQUET] first 5 rows (string columns):")
    print(peek)
else:
    print("ℹ️  No string (Utf8) columns present in the Parquet — no label columns inside.")

print("Parquet dimension checks — End")


In [ ]:
# ADD LABEL AND ID_TISSUE BETA <-> PHENO 287331
BETA_PATH   = Path("/kaggle/input/gse287331-parquet/GSE287331.parquet")
PHENO_PATH  = Path("/kaggle/input/pheno-gse287331/pheno_GSE287331.parquet")

# Output
OUT_PATH   = "/kaggle/working/GSE287331_ID_TISSUE.parquet"

ID_BETA_COL      = "id_tissue"       
LABEL_BETA_COL   = "label"           

ID_PHENO_MATCH   = "idat_basename"   
ID_PHENO_NEW     = "id_tissue"       
LABEL_PHENO_COL  = "label"           

# 1) LOAD PHENO (MAPPING TABLE)
print("[INFO] Loading pheno mapping…")
pheno = pl.read_parquet(
    PHENO_PATH,
    columns=[ID_PHENO_MATCH, ID_PHENO_NEW, LABEL_PHENO_COL],
)

print(f"[OK] pheno shape: {pheno.shape}")
print(pheno.head())

dup_counts = (
    pheno
    .group_by(ID_PHENO_MATCH)
    .len()
    .filter(pl.col("len") > 1)
)
if dup_counts.height > 0:
    print("⚠️  WARNING: some idat_basename have multiple rows in pheno!")
    print(dup_counts)
    # puoi decidere se fare .unique() o ispezionare
pheno_unique = pheno.unique(subset=[ID_PHENO_MATCH])

# 2) CHECK MATCH AND LABEL 
print("\n[INFO] Loading ONLY id_tissue + label from beta for checks…")
beta_meta = pl.read_parquet(
    BETA_PATH,
    columns=[ID_BETA_COL, LABEL_BETA_COL],
)
print(f"[OK] beta_meta shape: {beta_meta.shape}")
print(beta_meta.head())

# Join beta_meta (id_tissue) WITH pheno(idat_basename)
check_df = (
    beta_meta
    .join(
        pheno_unique,
        left_on=ID_BETA_COL,
        right_on=ID_PHENO_MATCH,
        how="left",
        suffix="_pheno"
    )
)

# 2a) ID WITHOUT match 
missing_in_pheno = check_df.filter(pl.col(ID_PHENO_NEW).is_null())
if missing_in_pheno.height > 0:
    print("\n❌ ERROR: some beta id_tissue have NO match in pheno[idat_basename]!")
    print(missing_in_pheno.select([ID_BETA_COL, LABEL_BETA_COL]).head(20))
    raise SystemExit("Stopping: fix pheno/beta ID mismatch before rewriting Parquet.")

# 2b) Label INCORRECT
mismatch_label = check_df.filter(pl.col(LABEL_BETA_COL) != pl.col(LABEL_PHENO_COL))
if mismatch_label.height > 0:
    print("\n❌ ERROR: label mismatch between beta and pheno!")
    print(mismatch_label.select([
        ID_BETA_COL,
        LABEL_BETA_COL,
        LABEL_PHENO_COL,
    ]).head(50))
    raise SystemExit("Stopping: fix label mismatches before rewriting Parquet.")
else:
    print("\n[CHECK] All labels are consistent between beta and pheno ✅")

# 3) MAPPING
print("\n[INFO] Building idat_basename → new id_tissue mapping…")

mapping_df = pheno_unique.select([ID_PHENO_MATCH, ID_PHENO_NEW])
old_ids = mapping_df[ID_PHENO_MATCH].to_list()
new_ids = mapping_df[ID_PHENO_NEW].to_list()

print(f"[OK] mapping size: {len(old_ids)} entries")

print("\n[INFO] Loading FULL beta Parquet to update id_tissue (this may take a bit)…")
beta_full = pl.read_parquet(BETA_PATH)
print(f"[OK] beta_full shape: {beta_full.shape}")

beta_updated = beta_full.with_columns(
    pl.col(ID_BETA_COL).replace(old_ids, new_ids)
)

# Check 
null_ids = beta_updated.filter(pl.col(ID_BETA_COL).is_null())
if null_ids.height > 0:
    print("\n❌ ERROR: some updated id_tissue are NULL after replacement!")
    print(null_ids.head())
    raise SystemExit("Stopping: something went wrong in mapping.")

print("\n[CHECK] id_tissue successfully updated for all rows ✅")

# 4) SAVE NEW PARQUET
print(f"\n[INFO] Writing updated beta Parquet to: {OUT_PATH}")
beta_updated.write_parquet(
    OUT_PATH,
    compression="lz4"   
)

print("[DONE] New Parquet written with updated id_tissue column")
